# Tier 2 baseline — windowed masquerade detector (Pi 5)

Trains a shallow Random Forest on the **100 ms window features** that
`gateway_bridge.py --collect` logged on the virtual bus, to separate normal
flight telemetry from the **timing-preserving masquerade** (`07_masquerade.py`):
same CAN ID, same 50 Hz cadence, impossible altitude payload while RPM stays flat.

This is the TinyML justification experiment: Tier 1 rules (ID / DLC / flood rate)
let every masquerade frame through — see `masquerade_failure.log` — so detection
needs temporal + cross-message features, i.e. a model in the gateway.

In [ ]:
import joblib, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from window_features import FEATURE_COLS, WINDOW_S

In [ ]:
NORMAL = "data/win_normal.csv"    # gateway --collect --label 0   (01_make_traffic only)
ATTACK = "data/win_attack.csv"    # gateway --collect --label 1   (01 --no-baro + 07 takeover)
SPIKE_M = 5.0                     # >5 m baro jump in one 100 ms window is physically
                                  # impossible -> safe ground truth for OUR injected spikes

benign = pd.read_csv(NORMAL).assign(label=0)
attack = pd.read_csv(ATTACK)
# EVENT-level labels: only windows that actually carry an injected spike are attack.
# Windows between spikes are honest by construction (with the default spike cadence
# ~9 of 10 attack-run windows are honest - run-level labels would be ~90% wrong).
attack["label"] = (attack["baro_dalt_max_m"] > SPIKE_M).astype(int)
df = pd.concat([benign, attack], ignore_index=True).fillna(0.0)
print(f"{len(df)} windows | normal={int((df.label == 0).sum())} "
      f"event={int((df.label == 1).sum())} (of {len(attack)} in the attack run)")
df.head(3)

In [ ]:
X, y = df[FEATURE_COLS].to_numpy("float32"), df["label"].to_numpy()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

clf = RandomForestClassifier(n_estimators=50, max_depth=8, n_jobs=-1, random_state=42)
clf.fit(X_tr, y_tr)

In [ ]:
pred = clf.predict(X_te)
print(classification_report(y_te, pred, target_names=["normal", "event"], digits=4))
print("Confusion matrix (rows=truth, cols=pred):")
print(confusion_matrix(y_te, pred))

# the detector PAIR - never report detection rate alone:
fp = int(((y_te == 0) & (pred == 1)).sum()); n_ben = int((y_te == 0).sum())
fn = int(((y_te == 1) & (pred == 0)).sum()); n_ev = int((y_te == 1).sum())
print(f"\nevent detection rate: {100 * (n_ev - fn) / max(n_ev, 1):.1f}% "
      f"({n_ev - fn}/{n_ev} lie-carrying windows caught)")
print(f"false alarms: {fp}/{n_ben} benign windows "
      f"= {fp / (n_ben * WINDOW_S / 3600):.1f} per benign hour")

In [ ]:
# Which temporal features carry the separation? (expect baro payload deltas +
# cross-message consistency near the top - exactly what Tier 1 cannot see)
pd.Series(clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False).head(8)

In [ ]:
joblib.dump(clf, "model_window.pkl")
print("saved model_window.pkl - live: python gateway_bridge.py --model model_window.pkl")

## Reading the result

- **Near-100% here is expected** (synthetic single-attack-class data on vcan) — report
  it as a *feasibility* baseline, not a field result. Committee-proofing notes are in
  README §3: hold out an attack class, and cross-test on the replayed UAVCAN/HCRL logs.
- **Labels are event-level**: only windows whose `baro_dalt_max_m` exceeds the
  physically-impossible threshold count as attack; windows between spikes are honest
  by construction. Report the **detector pair** — event detection rate AND false
  alarms per benign hour (measured on benign traffic only). A lone accuracy number
  is not a result.
- The features doing the work (`baro_dalt_max_m`, `baro_climb_mps`, `pdelta_max_baro`,
  `rpm_dmax`) are **window statistics, not frame fields** — that is the argument for
  putting a 100 ms inference window (INT8, C++) into the Phase 2 gateway.